# Projeto 4 — Clipping comparativo (peso 2)

**Pergunta:** o que este recorte de notícias da Agência Brasil e da CNN Brasil mostra sobre a cobertura de Lula e Flávio Bolsonaro? Transforme textos em dados auditáveis e escreva uma conclusão proporcional ao tamanho da amostra. A base tem 40 matérias selecionadas, dez por combinação de veículo e foco, publicadas em outubro de 2026. **Não é uma amostra aleatória de toda a cobertura.**

Entregue uma pasta `23-projeto-4-noticias/` no seu repositório privado com notebook executado, `README.md` e tabelas de resultado pequenas. O projeto integra Aulas 21–23 e vale peso 2. Crie sua **própria chave Groq**; nunca a salve nem a envie.

## 0. Carregar e conferir a base

Abra no Colab. Se este notebook estiver numa pasta isolada, ele baixa a tabela publicada no repositório da disciplina. Confira 40 linhas, datas, URLs e o equilíbrio 10×4.

In [ ]:
import pandas as pd
from pathlib import Path
arquivo_local = Path("dados/noticias_lula_flavio_2026.csv")
if not arquivo_local.exists():
    arquivo_local = Path("../dados/noticias_lula_flavio_2026.csv")
url_base = "https://raw.githubusercontent.com/mateuspestana/extracao_analise_2026/main/aulas/23-extracao-busca-semantica/dados/noticias_lula_flavio_2026.csv"
politica = pd.read_csv(arquivo_local if arquivo_local.exists() else url_base)
print(politica.shape)
print(pd.crosstab(politica["veiculo"], politica["foco_amostral"]))
assert len(politica) == 40

## 1. Preparação textual — Aula 21

Preserve `texto_analise`; crie `texto_preparado` com regras justificadas. Conte documentos vazios, duplicatas, tokens antes/depois. Mostre três casos lado a lado; um deve conter nome próprio ou negação.

In [ ]:
# Seu código e comentários aqui.

## 2. Tópicos — Aula 22

Aplique TF-IDF e NMF às 40 linhas. Use poucos tópicos e `random_state=42`. Mostre palavras e manchetes representativas, dê nomes provisórios e faça uma tabela por veículo e foco. Conte também as menções a Lula e Flávio nos títulos e resumos, sempre com o denominador de cada grupo. Explique ao menos um tópico confuso. As editorias do corpus geral da aula não existem nesta base.

In [ ]:
# Seu código e leitura crítica aqui.

## 3. Extração estruturada — Aula 23

Crie sua própria chave Groq. Reaproveite o esquema e a função `extrair()` da aula principal. Selecione **4 notícias por combinação de veículo e foco** com `groupby(...).sample(n=4, random_state=42)`; são 16 chamadas ao todo. Faça primeiro **uma** chamada de teste. Depois rode o lote **uma vez** e guarde a tabela. Não repita o lote ao ajustar gráficos ou texto do relatório.

Confira se `evidencia` é um trecho literal de `texto_analise`. Audite manualmente pelo menos quatro fichas, uma por combinação. Marque ausências, exageros e atribuições ambíguas; não conserte silenciosamente a saída do modelo.

In [ ]:
from getpass import getpass
chave_groq = getpass("Sua chave Groq (não aparece): ")
amostra_ia = politica.groupby(["veiculo", "foco_amostral"], group_keys=False).sample(n=4, random_state=42)
print(pd.crosstab(amostra_ia.veiculo, amostra_ia.foco_amostral))


In [ ]:
import requests, json
modelo_groq = "openai/gpt-oss-20b"
esquema = {
    "type": "object", "additionalProperties": False,
    "properties": {
        "pessoa_central": {"type": ["string", "null"]},
        "fato": {"type": "string"},
        "tema": {"type": "string"},
        "evidencia": {"type": "string"},
        "limite": {"type": "string"}
    },
    "required": ["pessoa_central", "fato", "tema", "evidencia", "limite"]
}

def extrair(texto):
    resposta = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {chave_groq}"},
        json={"model": modelo_groq, "temperature": 0, "reasoning_effort": "low", "max_completion_tokens": 600,
              "messages": [{"role": "system", "content": "Extraia apenas fatos presentes no texto recebido. A evidencia deve ser um trecho literal e contínuo do texto. Se não houver pessoa central, use null. Não infira intenção política nem viés do veículo."},
                           {"role": "user", "content": texto}],
              "response_format": {"type": "json_schema", "json_schema": {"name": "ficha_noticia", "strict": True, "schema": esquema}}},
        timeout=40)
    if resposta.status_code == 429:
        raise RuntimeError("Limite Groq atingido (429). Aguarde antes de tentar outra chamada.")
    resposta.raise_for_status()
    return json.loads(resposta.json()["choices"][0]["message"]["content"])


In [ ]:
# Teste uma notícia antes de rodar o lote.
teste = extrair(amostra_ia.iloc[0]["texto_analise"])
print(teste)
print("Evidência literal:", teste["evidencia"] in amostra_ia.iloc[0]["texto_analise"])


In [ ]:
# Execute uma vez após revisar a função e a chamada de teste:
# fichas = []
# for _, linha in amostra_ia.iterrows():
#     resultado = extrair(linha["texto_analise"])
#     fichas.append({"id": linha["id"], "veiculo": linha["veiculo"],
#                    "foco_amostral": linha["foco_amostral"], **resultado,
#                    "evidencia_literal": resultado["evidencia"] in linha["texto_analise"]})
# fichas = pd.DataFrame(fichas)
# fichas.to_csv("fichas_extraidas.csv", index=False)

## 4. Busca — Aula 23

Faça uma pergunta substantiva. Compare top 3 de busca por palavra e de busca semântica nas 40 matérias. Registre IDs, títulos, links e uma avaliação manual da relevância. A consulta precisa poder ser repetida.

In [ ]:
# Seu código e comparação aqui.

## 5. Relatório e entrega

No `README.md`, siga esta ordem: objetivo e pergunta; preparação; revisão de 21–22; roteiro de análise; prática aplicada; resultados do desafio; entrega; declaração de uso de IA; checklist. Inclua fonte, janela temporal, regra de seleção, tabela de denominadores, pelo menos dois achados com IDs/URLs, quatro fichas auditadas, tópico confuso, comparação de buscas e limitações. Separe **descrição dos dados** de **interpretação**. Não chame a diferença observada de “viés do jornal” sem um protocolo mais amplo.

Faça um commit de progresso e um de entrega. Prepare-se para explicar uma decisão de limpeza, um tópico, uma ficha e uma busca.

- [ ] 40 linhas: 10 por veículo × foco.
- [ ] Chave pessoal fora dos arquivos entregues.
- [ ] Extração em amostra de 16, com quatro auditorias.
- [ ] Conclusões citam IDs e links e respeitam o recorte.
- [ ] Notebook executa de cima a baixo no Colab.